In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,baseline,0,141,38.097624
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,baseline,0,141,26.094690
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,baseline,0,141,36.944095
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,baseline,0,141,45.143502
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,baseline,0,141,35.915273
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,0,129,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,0,129,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,0,129,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,0,129,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  2.486206e+06
              2                  2.584635e+06
              3                  2.252363e+06
              4                  2.009872e+06
              5                  1.530386e+06
intervention  1                  2.486209e+06
              2                  2.584639e+06
              3                  2.252368e+06
              4                  2.009879e+06
              5                  1.530392e+06
zero          1                  2.486206e+06
              2                  2.584635e+06
              3                  2.252363e+06
              4                  2.009872e+06
              5                  1.530386e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.312663e+06
              2                  1.328777e+06
              3                  1.122774e+06
              4                  1.022436e+06
              5                  6.945790e+05
intervention  1                  1.299541e+06
              2                  1.310140e+06
              3                  1.099306e+06
              4                  1.001721e+06
              5                  6.724681e+05
zero          1                  1.312663e+06
              2                  1.328777e+06
              3                  1.122774e+06
              4                  1.022436e+06
              5                  6.945790e+05
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.527978
              2                  0.514106
              3                  0.498487
              4                  0.508707
              5                  0.453859
intervention  1                  0.522700
              2                  0.506895
              3                  0.488067
              4                  0.498399
              5                  0.439409
zero          1                  0.527978
              2                  0.514106
              3                  0.498487
              4                  0.508707
              5                  0.453859
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,17452.628925
1,Female,0.0,0.019178,not_pregnant,2,17380.997496
2,Female,0.0,0.019178,not_pregnant,3,16409.683914
3,Female,0.0,0.019178,not_pregnant,4,14424.706826
4,Female,0.0,0.019178,not_pregnant,5,13407.888483
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,5056.112334
281,Male,95.0,125.000000,not_pregnant,2,4424.735951
282,Male,95.0,125.000000,not_pregnant,3,4548.244848
283,Male,95.0,125.000000,not_pregnant,4,4870.219344


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    2.372743e+06
2    2.463522e+06
3    2.073741e+06
4    1.684268e+06
5    1.470164e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.252757e+06
              2                  1.266512e+06
              3                  1.033733e+06
              4                  8.567992e+05
              5                  6.672468e+05
intervention  1                  1.240232e+06
              2                  1.248747e+06
              3                  1.012124e+06
              4                  8.394372e+05
              5                  6.460035e+05
zero          1                  1.252757e+06
              2                  1.266512e+06
              3                  1.033733e+06
              4                  8.567992e+05
              5                  6.672468e+05
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,baseline,0,141,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,baseline,0,141,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,baseline,0,141,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,baseline,0,141,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,baseline,0,141,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,0,129,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,0,129,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,0,129,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,0,129,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.560970e+06
              2                  2.158754e+06
              3                  2.072059e+06
              4                  2.032784e+06
              5                  1.396694e+06
intervention  1                  2.553233e+06
              2                  2.147785e+06
              3                  2.058858e+06
              4                  2.020190e+06
              5                  1.385664e+06
zero          1                  2.560970e+06
              2                  2.158754e+06
              3                  2.072059e+06
              4                  2.032784e+06
              5                  1.396694e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if pathlib.Path(path).is_file():
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,baseline,0,81,195.420449
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,baseline,0,81,193.791945
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,baseline,0,81,174.249900
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,baseline,0,81,149.822344
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,baseline,0,81,109.109751
...,...,...,...,...,...,...,...,...,...,...,...,...
23995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,71,153.079352
23996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,71,170.992893
23997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,71,143.308329
23998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,71,122.137781


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  174653.769261
              2                  180322.590785
              3                  158254.736585
              4                  139955.240042
              5                  106414.576984
intervention  1                  174640.741231
              2                  180303.048740
              3                  158228.680525
              4                  139927.555479
              5                  106395.034939
zero          1                  174653.769261
              2                  180322.590785
              3                  158254.736585
              4                  139955.240042
              5                  106414.576984
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,14165.730444,zero
1,Female,0.0,0.019178,2,13819.564387,zero
2,Female,0.0,0.019178,3,12897.953437,zero
3,Female,0.0,0.019178,4,11161.458690,zero
4,Female,0.0,0.019178,5,9883.361872,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,3765.420260,intervention
746,Male,95.0,125.000000,2,3327.310148,intervention
747,Male,95.0,125.000000,3,3461.357460,intervention
748,Male,95.0,125.000000,4,3713.841919,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.369123e+07
              2                  2.339761e+07
              3                  2.169052e+07
              4                  2.063809e+07
              5                  1.935449e+07
intervention  1                  2.347697e+07
              2                  2.310291e+07
              3                  2.130967e+07
              4                  2.019268e+07
              5                  1.887728e+07
zero          1                  2.369123e+07
              2                  2.339761e+07
              3                  2.169052e+07
              4                  2.063809e+07
              5                  1.935449e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.494398e+07
              2                  2.466412e+07
              3                  2.272426e+07
              4                  2.149489e+07
              5                  2.002174e+07
intervention  1                  2.471720e+07
              2                  2.435166e+07
              3                  2.232180e+07
              4                  2.103212e+07
              5                  1.952329e+07
zero          1                  2.494398e+07
              2                  2.466412e+07
              3                  2.272426e+07
              4                  2.149489e+07
              5                  2.002174e+07
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  7596.117757
              2                  7767.503194
              3                  7031.727783
              4                  6232.914051
              5                  5474.484244
baseline      1                  7596.117757
              2                  7767.503194
              3                  7031.727783
              4                  6232.914051
              5                  5474.484244
intervention  1                  7278.200535
              2                  7153.132715
              3                  6263.036641
              4                  5331.319634
              5                  4596.984541
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)